# Program Availability, Location, and Community Context in NYC

<figure>
  <img src="edprogram.jpg" alt="Students in an education program">
  <figcaption>
    Source: https://unsplash.com/photos/teacher-instructing-students-in-a-classroom-lecture-u6OsIM1ZEnk
  </figcaption>
</figure>

This dataset lists various NYC education program sites: each row is one program at a particular location. This analysis is useful for seeing where programs are, who runs them, and how they spread across boroughs and neighborhoods. Remember that some addresses or buildings appear more than once because they host several programs.


| Column | Definition |
|---|---|
| program type | Broad category of the service or initiative (for example, early childhood, after-school, youth employment), used to group programs for reporting and filtering. |
| program | The specific program title or service name as operated or funded by the listed agency. |
| site name | The public name of the physical location where the program runs (often a school, center, or campus). |
| borough/community| The NYC borough and/or community area associated with the site, for geographic and administrative context. |
| agency | The city department, nonprofit, or other organization responsible for operating or overseeing the program. |
| contact Number | Telephone number for the site or program, for scheduling, enrollment, or general inquiries. |
| grade level / age group  | The grades or age range the program is intended to serve (for example, PK–5, 6–8, or ages 14–21). |
| location 1 | Primary street address or free-text location description for the site (often used for mapping or mail). |
| postcode | ZIP (postal) code for the site’s address, used for geography and mail routing. |
| latitude | North–south geographic coordinate of the site, in decimal degrees, for mapping and distance analysis. |
| longitude | East–west geographic coordinate of the site, in decimal degrees, for mapping and distance analysis. |
| community board | NYC community board number for the site’s area; community boards are local advisory bodies within each borough. |
| community council  | Local representative body related to education or community governance in the area (naming varies by source; often aligned with school-community or council districts in the original dataset). |
| census tract | Small statistical area used by the U.S. Census Bureau; identifies the tract containing the site for demographic and equity analysis. |
| bin | NYC Building Identification Number: a unique identifier for a building as used by city agencies (for example, for permits and safety). |
| bbl | Borough–Block–Lot: NYC tax parcel identifier that pinpoints the property lot for the site. |
| nta | Neighborhood Tabulation Area: a Census-based neighborhood unit used in NYC planning and population summaries for the area around the site. |

In [165]:
# pip install -U kaleido

In [113]:
import plotly.io as pio
pio.renderers.default = "png"

In [114]:
#Import statement
import pandas as pd
import matplotlib.pyplot as plt 
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import re
# import spacy 

from scipy import stats


# pd.options.display.float_format = '{:,.2f}'.format
# !python -m spacy download en_core_web_sm

### Data Quality Checks 


In [115]:
#load dataset
df = pd.read_csv(r'C:\Users\Admin\Documents\Analysis excercises\Education NYC\schooldata.csv')

df.head(2)

,PROGRAM TYPE,PROGRAM,SITE NAME,BOROUGH / COMMUNITY,AGENCY,Contact Number,Grade Level / Age Group,Location 1,Postcode,Latitude,Longitude,Community Board,Community Council,Census Tract,BIN,BBL,NTA
0,"Reading & Writing,NDA Programs,Family Literacy",Adolescent Literacy,K 533- School for Democracy and Leadership 600...,Brooklyn,CAMBA,718.282.5575,grades 6 to 8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"After-School Programs,NDA Programs,Youth Educa...",High-School Aged Youth,Voyagees Prepatory High School,Queens,Central Brooklyn Economic Development Corporation,(718) 592-5757,High School,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [116]:
#list out all columns in the df
df.columns

Index(['PROGRAM TYPE', 'PROGRAM', 'SITE NAME', 'BOROUGH / COMMUNITY', 'AGENCY',
       'Contact Number', 'Grade Level / Age Group ', 'Location 1', 'Postcode',
       'Latitude', 'Longitude', 'Community Board', 'Community Council ',
       'Census Tract', 'BIN', 'BBL', 'NTA'],
      dtype='str')

In [117]:
#Create clean, consistent column names and/or rename

df.columns = df.columns.str.replace(r'\s+', ' ', regex=True).str.lower().str.strip().str.replace('/', '', regex=False).str.replace(' ', '_')


In [118]:
#possibly rename columns?
#.rename(columns=({'A':'B', 'C':'D'}))

df = df.rename(columns=({
    
    'grade_level__age_group': 'grade_level',
    'borough__community' : 'borough_community',
    'location_1' : 'location'
    
    
}))

In [119]:
df.columns

Index(['program_type', 'program', 'site_name', 'borough_community', 'agency',
       'contact_number', 'grade_level', 'location', 'postcode', 'latitude',
       'longitude', 'community_board', 'community_council', 'census_tract',
       'bin', 'bbl', 'nta'],
      dtype='str')

In [120]:
#check dimension of df 
df.shape

(464, 17)

In [121]:
#check if lat and long is in the right ranges
invalid_latlon = df[~(
    df['latitude'].between(-90, 90) &
    df['longitude'].between(-180, 180)
)]

In [122]:
invalid_latlon.shape
#drop these? if we had access to the correct ones, we couldve changed but otherwise, we drop. We dont need to have school with wrong lat-lon

(110, 17)

In [123]:
#unique entries in categorical columns 
df.program_type.unique() 

<StringArray>
[                                                            'Reading & Writing,NDA Programs,Family Literacy',
                                               'After-School Programs,NDA Programs,Youth Educational Support',
                                                                                'Family Support,NDA Programs',
                                                            'Immigration Services,Immigrant Support Services',
                                                           'NDA Programs,Senior Programs,Older Adult Program',
      'Reading & Writing,NDA Programs,English Language Program,Family Literacy,Adult Basic Education,ABE/GED',
                           'After-School Programs,NDA Programs,Adolescent Literacy,Youth Educational Support',
                               'Reading & Writing,NDA Programs,English Language Program,ESOL,Family Literacy',
 'Reading & Writing,NDA Programs,English Language Program,ESOL,Family Literacy,Adult Basic Educati

In [124]:
# Look at unique entries in each of categorical columns
import pandas as pd
from IPython.display import display, HTML


def value_counts_with_total(series: pd.Series, count_col="count", index_name="value") -> pd.DataFrame:
    """Counts per value, NaN shown as '(missing)', last row '(total rows)' = len(series)."""
    vc = series.value_counts(dropna=False)
    vc.index = vc.index.fillna("(missing)")
    body = vc.rename_axis(index_name).to_frame(count_col)
    total = pd.DataFrame(
        {count_col: [series.size]},
        index=pd.Index(["(total rows)"], name=index_name),
    )
    return pd.concat([body, total])


# Each (series, index_name) pair becomes one column; add or remove entries as needed.
categorical_summaries = [
    (df["site_name"], "site name"),
    (df["program"], "agency"),
    (df["borough_community"], "community"),
]

SCROLL_MAX_HEIGHT = "420px"  # per-column scroll area; adjust as needed

tables = [
    value_counts_with_total(s, index_name=name) for s, name in categorical_summaries
]
cells = []
for i, t in enumerate(tables):
    pad = "padding-right:24px;" if i < len(tables) - 1 else ""
    inner = f'<div style="max-height:{SCROLL_MAX_HEIGHT}; overflow:auto;">{t.to_html()}</div>'
    cells.append(f'<td style="vertical-align:top; {pad}">{inner}</td>')
display(HTML("<table><tr>" + "".join(cells) + "</tr></table>"))

,count
site name,
Make the Road New York,10
"Jewish Community Council of Greater Coney Island, Inc.",7
"Brooklyn Housing & Family Services, Inc.",6
Asian Americans for Equality,5
Citizens Advice Bureau,5
Northern Manhattan Improvement Corporation,4
"Council of Jewish Orgainzations of Flatbush, Inc.",4
The Jewish Community Center of Staten Island,4
The Children's Aid Society,4


In [125]:
# Look at unique entries in each of a subset of categorical columns
import pandas as pd
from IPython.display import display, HTML


def value_counts_with_total(series: pd.Series, count_col="count", index_name="value") -> pd.DataFrame:
    """Counts per value, NaN shown as '(missing)', last row '(total rows)' = len(series)."""
    vc = series.value_counts(dropna=False)
    vc.index = vc.index.fillna("(missing)")
    body = vc.rename_axis(index_name).to_frame(count_col)
    total = pd.DataFrame(
        {count_col: [series.size]},
        index=pd.Index(["(total rows)"], name=index_name),
    )
    return pd.concat([body, total])


# Each (series, index_name) pair becomes one column; add or remove entries as needed.
categorical_summaries = [
    (df["program_type"], "program_type"),
    (df["agency"], "agency"),
    (df["grade_level"], "grade level"),
]

SCROLL_MAX_HEIGHT = "420px"  # per-column scroll area; adjust as needed

tables = [
    value_counts_with_total(s, index_name=name) for s, name in categorical_summaries
]
cells = []
for i, t in enumerate(tables):
    pad = "padding-right:24px;" if i < len(tables) - 1 else ""
    inner = f'<div style="max-height:{SCROLL_MAX_HEIGHT}; overflow:auto;">{t.to_html()}</div>'
    cells.append(f'<td style="vertical-align:top; {pad}">{inner}</td>')
display(HTML("<table><tr>" + "".join(cells) + "</tr></table>"))

,count
program_type,
"Family Support,NDA Programs",132
"Immigration Services,Immigrant Support Services",68
"NDA Programs,Senior Programs,Older Adult Program",55
"After-School Programs,NDA Programs,Youth Educational Support",51
"Reading & Writing,NDA Programs,English Language Program,ESOL,Family Literacy",46
"After-School Programs,NDA Programs,Adolescent Literacy,Youth Educational Support",42
"Reading & Writing,NDA Programs,Family Literacy",28
"Reading & Writing,NDA Programs,English Language Program,Family Literacy,Adult Basic Education,ABE/GED",14
"NDA Programs,Immigration Services,Immigrant Support Services",11


In [126]:
#check datatypes 
df.dtypes

program_type             str
program                  str
site_name                str
borough_community        str
agency                   str
contact_number           str
grade_level              str
location                 str
postcode             float64
latitude             float64
longitude            float64
community_board      float64
community_council    float64
census_tract         float64
bin                  float64
bbl                  float64
nta                      str
dtype: object

In [127]:
#check missing values in entire df.
#need to deal with these
df.isnull().values.any()



np.True_

In [128]:
#check missing values in each column.

df.isnull().sum()

program_type           0
program                0
site_name              0
borough_community      0
agency                 0
contact_number         1
grade_level           54
location               6
postcode             110
latitude             110
longitude            110
community_board      110
community_council    110
census_tract         110
bin                  117
bbl                  117
nta                  110
dtype: int64

In [129]:
#check for duplicates
df[df.duplicated()]

,program_type,program,site_name,borough_community,agency,contact_number,grade_level,location,postcode,latitude,longitude,community_board,community_council,census_tract,bin,bbl,nta
55,"Family Support,NDA Programs",Housing,"Brooklyn Housing & Family Services, Inc.",Brooklyn,"Council of Jewish Orgainzations of Flatbush, Inc.",(718) 435-7585,Adults,"415 Albemarle Road\nBrooklyn, NEW YORK 11218\n...",11218.0,40.64551,-73.976296,12.0,39.0,500.0,3124269.0,3.053270e+09,Windsor Terrace ...


In [130]:
#check duplicates 
df.duplicated().value_counts()



False    463
True       1
Name: count, dtype: int64

In [131]:
#check nuanced duplicates. Start with subset of numerical columns and then subset of categorical columns
#.value_counts() is quite useful.
df.columns

df.duplicated(subset=['program_type', 'program', 'site_name']).value_counts()


False    420
True      44
Name: count, dtype: int64

In [132]:
df.duplicated(subset=['latitude', 'longitude']).value_counts()

False    280
True     184
Name: count, dtype: int64

In [133]:
#check info() if necessary.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 464 entries, 0 to 463
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   program_type       464 non-null    str    
 1   program            464 non-null    str    
 2   site_name          464 non-null    str    
 3   borough_community  464 non-null    str    
 4   agency             464 non-null    str    
 5   contact_number     463 non-null    str    
 6   grade_level        410 non-null    str    
 7   location           458 non-null    str    
 8   postcode           354 non-null    float64
 9   latitude           354 non-null    float64
 10  longitude          354 non-null    float64
 11  community_board    354 non-null    float64
 12  community_council  354 non-null    float64
 13  census_tract       354 non-null    float64
 14  bin                347 non-null    float64
 15  bbl                347 non-null    float64
 16  nta                354 non-null    st

In [134]:
#numerical columns
df.describe()

,postcode,latitude,longitude,community_board,community_council,census_tract,bin,bbl
count,354.000000,354.000000,354.000000,354.000000,354.000000,354.000000,3.470000e+02,3.470000e+02
mean,10903.118644,40.720120,-73.932879,7.022599,27.593220,5507.745763,2.927819e+06,2.836953e+09
std,495.951351,0.083889,0.068352,4.654590,14.174891,17535.647323,1.087317e+06,1.044325e+09
min,10002.000000,40.565262,-74.183882,1.000000,1.000000,2.000000,1.001162e+06,1.000748e+09
25%,10457.000000,40.649128,-73.976296,3.000000,16.250000,159.750000,2.074718e+06,2.031495e+09
50%,11210.500000,40.705297,-73.935882,6.000000,28.500000,286.000000,3.083931e+06,3.027590e+09
75%,11231.000000,40.785838,-73.891593,12.000000,39.000000,578.000000,3.379620e+06,3.086930e+09
max,11691.000000,40.893523,-73.751962,27.000000,51.000000,105804.000000,5.077603e+06,5.059000e+09


In [135]:
#check describe(include='all', or 'str')

df.describe(include='str')


,program_type,program,site_name,borough_community,agency,contact_number,grade_level,location,nta
count,464,464,464,464,464,463,410,458,354
unique,11,21,383,18,312,252,15,399,111
top,"Family Support,NDA Programs",Adult Literacy,Make the Road New York,Brooklyn,Make the Road New York,(718) 542-0006,16+,"415 Albemarle Road\nBrooklyn, NEW YORK 11218\n...",Sunset Park East ...
freq,132,67,10,170,17,21,67,4,13


#### Summary of describe()
Missing values:

- contact_number has 463 non-null (so 1 missing).
- grade_level has 410 non-null (so 54 missing).
- location has 458 non-null (so 6 missing).

High-cardinality columns (lots of unique values):

- site_name 383 unique out of 464 rows → almost one per row (not great for grouping; use top‑N)
- agency 312 unique → also very high; consider bucketing (top‑N + “Other”) or acronyms
- location 399 unique → essentially unique addresses; better to extract borough/ZIP/lat/lon for analysis.
- borough_community shows 18 unique—but NYC should be 5.

### Data Cleaning

In [136]:
#make sure column names are proper 
df.columns

Index(['program_type', 'program', 'site_name', 'borough_community', 'agency',
       'contact_number', 'grade_level', 'location', 'postcode', 'latitude',
       'longitude', 'community_board', 'community_council', 'census_tract',
       'bin', 'bbl', 'nta'],
      dtype='str')

In [137]:
#Fix datatypes 

df.community_board = df['community_board'].astype('Int64')
df.grade_level = df['grade_level'].astype('string')
df.latitude = pd.to_numeric (df['latitude'], errors='coerce')
df.longitude = pd.to_numeric (df['longitude'], errors='coerce')


In [138]:
#drop invalid_latlon rows
df = df.drop(index= invalid_latlon.index)

In [139]:
df = df.drop_duplicates()

In [140]:
#drop duplcated rows by lat and lon. We dont need multiple schools with the same lat-lon. Better coords need to be provided for those
common_latlon = df.duplicated(subset=['latitude', 'longitude'])
df.drop(common_latlon.index)

,program_type,program,site_name,borough_community,agency,contact_number,grade_level,location,postcode,latitude,longitude,community_board,community_council,census_tract,bin,bbl,nta


In [141]:
df.shape

(353, 17)

In [142]:
df.isnull().sum()

#After some cleaning and dropping a few rows, we see that there are no null value except in the grade_level, bin asnd bbl. 
# We're not doing any calc on these columns so we can leave as is

program_type          0
program               0
site_name             0
borough_community     0
agency                0
contact_number        0
grade_level          42
location              0
postcode              0
latitude              0
longitude             0
community_board       0
community_council     0
census_tract          0
bin                   7
bbl                   7
nta                   0
dtype: int64

In [143]:
#standardise location column
df.location.unique()

<StringArray>
[       '95 05 Roosevelt Ave\nQueens, NEW YORK 11372\n(40.751630739367, -73.883612545283)',
          '90 25 161st Street\nQueens, NEW YORK 11432\n(40.71311535449, -73.792820999789)',
         '411 Pearl Street\nNew York, NEW YORK 10001\n(40.711348522879, -74.000800582114)',
        '1600 Central Avenue\nQueens, NEW YORK 11691\n(40.605264212936, -73.751891714766)',
 '216 Ft Washington Avenue\nNew York, NEW YORK 10032\n(40.842343769667, -73.942200758685)',
    '1720 Metropolitan Avenue\nBronx, NEW YORK 10462\n(40.840552520607, -73.853865592913)',
         '7802 Bay Parkway\nBrooklyn, NEW YORK 11214\n(40.606424326706, -73.989192742721)',
 '1460 Pennsylvania Avenue\nBrooklyn, NEW YORK 11239\n(40.646077795287, -73.879950181902)',
       '726 Stanley Avenue\nBrooklyn, NEW YORK 11207\n(40.660160583971, -73.882023470829)',
    '800 VAN SICLEN AVENUE\nBrooklyn, NEW YORK 11207\n(40.660221698125, -73.885485457846)',
 ...
      '88 Visitation Place\nBrooklyn, NEW YORK 11231\n(40.679

In [144]:
# pip install spacy
# python -m spacy download en_core_web_sm

import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")

def extract_places(text):
    if pd.isna(text):
        return pd.NA
    doc = nlp(str(text))
    places = [ent.text for ent in doc.ents if ent.label_ in {"GPE", "LOC"}]
    return ", ".join(dict.fromkeys(places))  # de-dupe, keep order

df["location_clean"] = df["location"].map(extract_places)

In [145]:
df.location_clean.unique()
#This is much better

<StringArray>
[                           'NEW YORK',                    'Queens, NEW YORK',
                  'New York, NEW YORK',            'Central Avenue, NEW YORK',
                  'Brooklyn, NEW YORK',                 'Manhattan, NEW YORK',
           'Jackson Heights, NEW YORK',                     'Bronx, NEW YORK',
          'Avenue, Brooklyn, NEW YORK',                   'Jamaica, NEW YORK',
                                    '',             'Northern Blvd, NEW YORK',
                    'Avenue, NEW YORK',   'Essex Street, Manhattan, NEW YORK',
   'Avenue, Jackson Heights, NEW YORK',        'Southern Boulevard, NEW YORK',
 'Thornton Street, Brooklyn, NEW YORK',    'Adams Street\nBrooklyn, NEW YORK',
     '21st Avenue, Brooklyn, NEW YORK',           'Avenue, Astoria, NEW YORK',
         'Fox Street, Bronx, NEW YORK',      'Southern Blvd\nBronx, NEW YORK',
         'Avenue, Manhattan, NEW YORK',  'Wyckoff Avenue, Brooklyn, NEW YORK',
       '21st Street, Queens, NEW YORK'

In [146]:
#standardise agency column

manual = {
    "CAMBA": "CAMBA",
    "HANAC, INC.": "HANAC",
    "F.E.G.S Brooklyn Resource Center": "FEGS",
    "Safe Horizon, Inc.": "Safe Horizon",
    "Manhattan Legal Services": "MLS",
    "Central Brooklyn Economic Development Corporation": "CBEDC",
    "East Harlem Employment Services, Inc.": "EHES",
    "Asian Americans for Equality": "AAFE",
    "St. Luke A.M.E Church": "St Luke AME",
    "Sanctuary for Families, Inc.": "SFF",
}

# words we don't want in acronyms
STOP = {
    "and","of","for","the","a","an","to","in","on","at","by","with",
    "inc","inc.","llc","ltd","co","corp","corporation","company",
    "services","service","center","school","academy","programs","program",
}

def auto_acronym(name: str) -> str:
    if pd.isna(name):
        return pd.NA

    s = str(name).strip()

    # normalize punctuation/spaces
    s = re.sub(r"[’']", "", s)
    s = re.sub(r"[^A-Za-z0-9\s]", " ", s)   # drop punctuation to spaces
    s = re.sub(r"\s+", " ", s).strip()

    # if already short-ish (e.g. CAMBA), keep
    if len(s) <= 8 and " " not in s:
        return s.upper()

    words = [w for w in s.split() if w and w.lower() not in STOP]

    # keep tokens like "I.S." / "MS" / numbers out of acronym unless we want them
    letters = []
    for w in words:
        if w.isdigit():
            continue
        # keep leading letter
        letters.append(w[0].upper())

    ac = "".join(letters)[:8]  # cap length
    return ac if ac else s[:12]  # fallback

# 2) create a clean key, then map
agency_clean = (
    df["agency"].astype("string")
      .str.strip()
      .str.replace(r"\s+", " ", regex=True)
)

df["agency_clean"] = agency_clean.map(lambda x: manual.get(str(x), auto_acronym(x)))

In [147]:
df.agency_clean.unique()

<StringArray>
[   'AAFE',      'SS',   'BSPAS',    'FDCP',     'NMI',   'RBSCC',      'PO',
   'IACRL',    'YMPJ',    'IHBC',
 ...
    'CATC',     'MLS',       'M', 'ISASVPA',   'HANAC', 'JMNRNYA',    'FEGS',
 'ECMJCHB',     'USA',   'CASEH']
Length: 228, dtype: str

In [148]:

#standardise program type column
df.program_type.unique()

<StringArray>
[                                                                               'Family Support,NDA Programs',
                                                            'Immigration Services,Immigrant Support Services',
                                               'After-School Programs,NDA Programs,Youth Educational Support',
      'Reading & Writing,NDA Programs,English Language Program,Family Literacy,Adult Basic Education,ABE/GED',
                                                           'NDA Programs,Senior Programs,Older Adult Program',
                           'After-School Programs,NDA Programs,Adolescent Literacy,Youth Educational Support',
                               'Reading & Writing,NDA Programs,English Language Program,ESOL,Family Literacy',
 'Reading & Writing,NDA Programs,English Language Program,ESOL,Family Literacy,Adult Basic Education,ABE/GED',
                                               'NDA Programs,Immigration Services,Immigrant Suppor

In [149]:

import pandas as pd

short_map = {
    "Reading & Writing,NDA Programs,English Language Program,Family Literacy,Adult Basic Education,ABE/GED": "Literacy program",
    "Family Support,NDA Programs": "Family support program",
    "Reading & Writing,NDA Programs,English Language Program,ESOL,Family Literacy": "Literacy program",
    "After-School Programs,NDA Programs,Adolescent Literacy,Youth Educational Support": "Youth program",
    "After-School Programs,NDA Programs,Youth Educational Support": "Youth program",
    "Immigration Services,Immigrant Support Services": "Immigration program",
    "Reading & Writing,NDA Programs,English Language Program,ESOL,Family Literacy,Adult Basic Education,ABE/GED": "Literacy program",
    "NDA Programs,Senior Programs,Older Adult Program": "Older Adults program",
    "NDA Programs,Immigration Services,Immigrant Support Services": "Immigration program",
    "Immigration Services,Immigrant Support Services,NDA Programs": "Immigration program",
    "Reading & Writing,NDA Programs,Family Literacy": "Literacy program",
}

def norm_cell(val):
    if pd.isna(val):
        return pd.NA
    parts = [p.strip() for p in str(val).split(",") if p.strip()]
    return ",".join(parts)  # normalize spaces consistently

df["program_type_clean"] = df["program_type"].map(norm_cell).replace(short_map)

df.program_type_clean.unique()

<StringArray>
['Family support program',    'Immigration program',          'Youth program',
       'Literacy program',   'Older Adults program']
Length: 5, dtype: str

In [150]:
#Standardize categories (e.g., spelling, capitalization, remove special characters)
df['program_type_clean'].str.title().str.strip().str.capitalize()

8      Family support program
9         Immigration program
11              Youth program
12     Family support program
13           Literacy program
                ...          
456    Family support program
457             Youth program
459          Literacy program
461          Literacy program
462      Older adults program
Name: program_type_clean, Length: 353, dtype: str

In [151]:
#standaradise borough community column
#In str.replace, the replacement (repl) must be a string (or a function), not np.nan.
import pandas as pd

s = df["borough_community"].astype("string").str.strip()

# map borough to proper admin district
fix = {
    "New York": "Manhattan",
    "Woodside": "Queens",
    "Jamaica": "Queens",
    "Corona": "Queens",
    "Kew Gardens": "Queens",
    "Jackson Heights": "Queens",
    "Astoria": "Queens",
    "Flushing": "Queens",
    "Forest Hills": "Queens",
    "Long Island City": "Queens",
}

df["borough_community"] = (
    s.mask(s.str.contains(r"\d", na=False), pd.NA)   # if it has a digit -> NA
     .str.replace(r"\s+", " ", regex=True)
     .str.title()
     .replace(fix)
     .fillna("Unknown")
)

df.borough_community.unique()
#perfect, NYC should only have 5

<StringArray>
['Queens', 'Manhattan', 'Bronx', 'Brooklyn', 'Staten Island']
Length: 5, dtype: string

In [152]:

cleaned = (
    df["contact_number"].astype("string").str.lower()
 .str.replace(r"[^a-z0-9\s]", "", regex=True)
    .str.replace(r"\b(?:ext|x)\s*\d+\b", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


parts = cleaned.str.split(r"\s+or\s+", n=1, regex=True, expand=True)
df["contact_number"] = parts[0].str.replace(r"\D", "", regex=True)


In [153]:
#standardise grade level column

import numpy as np
import pandas as pd

s = df["grade_level"].astype('string').str.strip().str.lower()

def standardise_grade(x: str):
    if pd.isna(x) or x == "":
        return pd.NA

    # broad buckets first
    if "all ages" in x:
        return "All ages"
    if "senior" in x:
        return "Seniors"
    if "adult" in x or "esol" in x or "civics" in x:
        return "Adults"
    if "parent" in x:
        return "Parents"
    if "high school" in x:
        return "High school"
    if "middle school" in x or "grades 6 to 8" in x or "grade 6 to 8" in x:
        return "Middle school"

    # age-style labels
    if "16 to 24" in x or "16+" in x or "18+" in x or "11+" in x:
        return "Young adults"
 

    #Other 
    if x in {"20-may", "14-nov"}:
        return "Unknown (needs review)"

    return "Other"

df["grade_level_clean"] = s.map(standardise_grade)

# quick check
df["grade_level_clean"].value_counts(dropna=False)



grade_level_clean
Young adults              72
Adults                    56
High school               43
NaN                       42
Seniors                   41
Middle school             41
All ages                  37
Parents                   14
Unknown (needs review)     7
Name: count, dtype: int64

In [154]:
#standardise site name column.
df.site_name.unique()


<StringArray>
[                              'Asian Americans for Equality',
              'Safe Space - Family Resource Center (Jamaica)',
                                        'Murray Bergtraum HS',
                               'Aids Center of Queens County',
                            'New Heights Neighborhood Center',
                             'St. Raymond Community Outreach',
 'Edith and Carl Marks Jewish Community House of Bensonhurst',
  'St. Dominic School / Italian American Civil Rights League',
                             'Boulevard Houses Senior Center',
                             'JHS 166 GEORGE GERSHWIN (K166)',
 ...
                     'Q 705- The Renaissance Charter School ',
                       'PSS/WSF GrandParent Family Apartment',
                                   'Far Rockaway High School',
                        'BCA Sunset Park Asian Senior Center',
                                          'Middle School 206',
                           'IS 204 -

In [155]:

s = df["site_name"].astype("string")
df["site_name_clean"] = (
    s.str.strip()
     .str.replace(r"\s+", " ", regex=True)
     .str.replace(r"[’']", "", regex=True)
     .str.replace(r"\s*-\s*", " - ", regex=True)
     .str.replace(r"\s*/\s*", " / ", regex=True)
)

In [156]:
from site_name_cleaning import add_site_name_clean

df = add_site_name_clean(df)

In [157]:
df["site_name_clean"].value_counts()

site_name_clean
Not clear             207
Center                 70
Primary_school         28
Some kind ofschool     25
High_school            18
Academy                 2
college                 2
Charter school          1
Name: count, dtype: int64

In [158]:
#Decide how to handle missing values (drop, fill, or leave as-is)
df.isna().sum()
#we dont have access to the grade_levels that are missing so its best to leave as is


program_type           0
program                0
site_name              0
borough_community      0
agency                 0
contact_number         0
grade_level           42
location               0
postcode               0
latitude               0
longitude              0
community_board        0
community_council      0
census_tract           0
bin                    7
bbl                    7
nta                    0
location_clean         0
agency_clean           0
program_type_clean     0
grade_level_clean     42
site_name_clean        0
dtype: int64

In [159]:
#quick check on new columns
df.columns

Index(['program_type', 'program', 'site_name', 'borough_community', 'agency',
       'contact_number', 'grade_level', 'location', 'postcode', 'latitude',
       'longitude', 'community_board', 'community_council', 'census_tract',
       'bin', 'bbl', 'nta', 'location_clean', 'agency_clean',
       'program_type_clean', 'grade_level_clean', 'site_name_clean'],
      dtype='str')

In [160]:
#arrange columns in proper order
cols = [
    "program",
    "program_type",
    "program_type_clean",
    "site_name",
    "site_name_clean",
    "borough_community",
    "agency",
    "agency_clean",
    "contact_number",
    "grade_level",
    "grade_level_clean",
    "location",
    "location_clean",
    "postcode",
    "latitude",
    "longitude",
    "community_board",
    "community_council",
    "census_tract",
    "bin",
    "bbl",
    "nta"
    

]
df = df[cols]

In [161]:
#clean df
df.head()

,program,program_type,program_type_clean,site_name,site_name_clean,borough_community,agency,agency_clean,contact_number,grade_level,...,location_clean,postcode,latitude,longitude,community_board,community_council,census_tract,bin,bbl,nta
8,Health Stat,"Family Support,NDA Programs",Family support program,Asian Americans for Equality,Not clear,Queens,Asian Americans for Equality,AAFE,7189610888,All Ages,...,NEW YORK,11372.0,40.748936,-73.871635,3,21.0,273.0,4036629.0,4.014830e+09,Jackson Heights ...
9,Domestic Violence Program,"Immigration Services,Immigrant Support Services",Immigration program,Safe Space - Family Resource Center (Jamaica),Center,Queens,Safe Space,SS,7187859062,All Ages,...,"Queens, NEW YORK",11432.0,40.705199,-73.799253,12,24.0,44601.0,4442231.0,4.097600e+09,Jamaica ...
11,High-School Aged Youth,"After-School Programs,NDA Programs,Youth Educa...",Youth program,Murray Bergtraum HS,Not clear,Manhattan,BCA Sunset Park Asian Senior Center,BSPAS,2127481225,High School,...,"New York, NEW YORK",10038.0,40.711442,-74.000851,1,1.0,29.0,1001388.0,1.001130e+09,Chinatown ...
12,Healthy Families,"Family Support,NDA Programs",Family support program,Aids Center of Queens County,Center,Queens,Father David Cassella Plaza,FDCP,7188962500,<NA>,...,"Central Avenue, NEW YORK",11691.0,40.605182,-73.751962,14,31.0,103202.0,4297967.0,4.155370e+09,Far Rockaway-Bayswater ...
13,Adult Literacy,"Reading & Writing,NDA Programs,English Languag...",Literacy program,New Heights Neighborhood Center,Center,Manhattan,Northern Manhattan Improvement Corporation,NMI,2128228300,16+,...,"New York, NEW YORK",10032.0,40.842571,-73.942097,12,10.0,251.0,1063381.0,1.021380e+09,Washington Heights South ...


### Exploratory Analysis
- This is where you get concrete answers to your research questions

In [162]:
#How many borough and program types are there overall

print(f'Number of unqiue borough: {df.borough_community.nunique()}')
print(f'Number of unqiue programs_types: {df.program_type_clean.nunique()}')

Number of unqiue borough: 5
Number of unqiue programs_types: 5


In [163]:
#Which borough has the most programs overall, and by program type?

q = df.groupby(['borough_community', 'program_type_clean'])['program'].size().sort_values(ascending=False).reset_index().head(30)

q


,borough_community,program_type_clean,program
0,Brooklyn,Family support program,44
1,Brooklyn,Youth program,36
2,Brooklyn,Literacy program,35
3,Brooklyn,Immigration program,23
4,Queens,Literacy program,21
5,Queens,Family support program,18
6,Bronx,Youth program,18
7,Queens,Immigration program,17
8,Bronx,Older Adults program,17
9,Bronx,Family support program,14


In [164]:
fig = px.bar(
    q,
    x='borough_community',
    y='program',
    barmode='group',
    color='program_type_clean',
    title='No. of programs by borough and type',
    # color_continous_scale='algae'
)
fig.update_layout(
    xaxis_title='Borough',
    yaxis_title='Frequency of programs'
)
# fig.update_yaxes(
#     autorange='reversed'
# )
fig.show(renderer="png")

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


In [ ]:
# top 10 agencies per borough 
#for each borough, agencies ordered from most to least rows
top_agencies = (
    df.groupby(["borough_community", "agency_clean"])
      .size()
      .reset_index(name="n")
      .sort_values(["borough_community", "n"], ascending=[True, False])
)

top_agencies.groupby("borough_community").head(5)

,borough_community,agency_clean,n
4,Bronx,CAB,8
6,Bronx,CAS,2
19,Bronx,HCL,2
21,Bronx,HF,2
31,Bronx,MM,2
106,Brooklyn,JCCGCI,7
122,Brooklyn,MRNY,5
57,Brooklyn,BCAA,4
126,Brooklyn,NYLAGN,4
158,Brooklyn,SYYBMB,4


In [ ]:
import plotly.express as px

plot_df = top_agencies.groupby("borough_community").head(10)

fig = px.bar(
    plot_df,
    x="n",
    y="agency_clean",
    color="borough_community",
    facet_col="borough_community",
    facet_col_wrap=2,
    orientation="h",
    height=800,
    title="Top 10 most frequent agency observed in each borough",
)
fig.update_yaxes(
    matches=None,
    autorange='reversed'
    )  # let each facet have its own agency labels
fig.show(renderer="png")

In [ ]:
#what are the distinct sites that run these programs?
counts = df["site_name_clean"].value_counts()
fig = px.pie(names=counts.index, values=counts.values, title="Distribution")
fig.update_traces(hole=0.4)  # donut hole (0 = pie, closer to 1 = thicker hole)
fig.show(renderer="png")

In [ ]:
 #How do grade level catgeories compare in frequency?
 r = df.groupby('grade_level_clean').size().sort_values(ascending=False).reset_index(name='count')
 r

,grade_level_clean,count
0,Young adults,72
1,Adults,56
2,High school,43
3,Middle school,41
4,Seniors,41
5,All ages,37
6,Parents,14
7,Unknown (needs review),7


In [ ]:
#Grade level frequency 

fig = px.bar(
    r,
    x='grade_level_clean',
    y='count',
    color='count',
    color_continuous_scale='algae',
    title='Grade Level frequency in this dataset'

)
fig.show(renderer="png")
#We can see that young adult education programs are the more prevalent in NYC

From the findings, we can conclude the following:
- There are five unique borough's - as expected for NYC

- After cleaning, there are 5 unique program types in this dataset.Those include:
    - Family support program,
    - Immigration program,
    - Literacy program,
    - Older Adults program, 
    - Youth program

- Brooklyn appears to have the highest concentration of program records in the top combinations, especially for Family support programs and Literacy program

- The “most frequent” agencies differ by borough, but a few show up strongly:
    - Bronx: CAB leads.
    - Brooklyn: JCCGCI leads; MRNY next.
    - Manhattan: NMI leads.
    - Queens: MRNY leads.
    - Staten Island: JCSI and MRNY lead.

- Grade level frequency (bar chart)
    - This dataset is most heavily represented by Young adults and Adults
    - Next tier: High school, Middle school, Seniors, All ages.
    - Smaller: Parents, Unknown (needs review).

